In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from langchain_groq import ChatGroq
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from firecrawl import Firecrawl
from langchain_community.document_loaders import PyPDFLoader
from langchain_experimental.text_splitter import SemanticChunker

C:\Users\Admin\AppData\Local\Temp\ipykernel_23496\2819652121.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
C:\Users\Admin\AppData\Local\Temp\ipykernel_23496\2819652121.py:6: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


In [ ]:
pdfloader = PyPDFLoader("C:\\Users\\Admin\\Desktop\\aai\\data\\hoticulture.pdf")
pdfdocs = pdfloader.load()

In [13]:
import os
from firecrawl.types import ScrapeOptions

firecrawl_api_key = os.getenv("FIRECRAWL_API_KEY")
if not firecrawl_api_key:
    raise RuntimeError("FIRECRAWL_API_KEY is missing.")

firecrawl = Firecrawl(api_key=firecrawl_api_key)

start_url = "https://agritech.tnau.ac.in/crop_protection/crop_prot_crop_diseases_agri.html"

# Increase the page limit. This consumes more Firecrawl credits.
webloader = firecrawl.crawl(
    start_url,
    limit=1000,
    max_discovery_depth=10,
    allow_external_links=False,
    scrape_options=ScrapeOptions(
        formats=["markdown"],
        only_main_content=False,
    ),
)

pages = getattr(webloader, "data", [])
print(f"Crawled pages: {len(pages)}")

if len(pages) >= 1000:
    print("WARNING: The crawl reached the configured limit.")

for index, page in enumerate(pages, start=1):
    metadata = page.metadata.model_dump(exclude_none=True)
    url = metadata.get("url", "unknown URL")
    content = (page.markdown or "").strip()

    print(f"{index}. {url} - {len(content)} characters")

    if not content:
        print("   WARNING: Empty page content")

Crawled pages: 214
1. https://agritech.tnau.ac.in/crop_protection/crop_prot_crop_diseases_agri.html - 8906 characters
2. https://agritech.tnau.ac.in/crop_protection/crop_prot_crop%20diseases_cereals_rice_main.html - 5123 characters
3. https://agritech.tnau.ac.in/crop_protection/crop_prot_crop%20diseases_cereals_sorghum.html - 3884 characters
4. https://agritech.tnau.ac.in/crop_protection/crop_prot_crop%20diseases_cereals_maize.html - 3040 characters
5. https://agritech.tnau.ac.in/crop_protection/crop_prot_crop_insect-veg_chillies.html - 3135 characters
6. https://agritech.tnau.ac.in/crop_protection/crop_prot_crop%20diseases_cereals_cumbu.html - 3650 characters
7. https://agritech.tnau.ac.in/crop_protection/crop_prot_crop%20diseases_cereals_wheat_main.html - 3384 characters
8. https://agritech.tnau.ac.in/crop_protection/crop_prot_crop%20diseases_cereals_ragi_main.html - 4375 characters
9. https://agritech.tnau.ac.in/crop_protection/crop_prot_crop_insect-veg_tomato.html - 3999 characters

In [8]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatGroq(model="qwen/qwen3.8-27b")

In [9]:
splitter = SemanticChunker(
    embeddings = embeddings,    
    breakpoint_threshold_type = "percentile" 
) 


In [14]:
from langchain_core.documents import Document

webdocs = []

for page in webloader.data:
    markdown = (page.markdown or "").strip()
    if not markdown:
        continue

    metadata = page.metadata.model_dump(exclude_none=True)

    webdocs.append(
        Document(
            page_content=markdown,
            metadata={
                **metadata,
                "source_type": "web",
                "source": metadata.get("url", "unknown"),
            },
        )
    )

for document in pdfdocs:
    document.metadata["source_type"] = "pdf"
    document.metadata["source"] = document.metadata.get(
        "source",
        r"C:\Users\Admin\Desktop\aai\data\hoticulture.pdf",
    )

all_documents = pdfdocs + webdocs
chunked_documents = splitter.split_documents(all_documents)

# Use a new collection name when rebuilding the index.
vectorstore = Chroma.from_documents(
    documents=chunked_documents,
    embedding=embeddings,
    collection_name="agri_rag_v2",
    persist_directory="./chroma_db",
)

retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 6,
        "fetch_k": 30,
        "lambda_mult": 0.7,
    },
)

print(f"Source documents: {len(all_documents)}")
print(f"Indexed chunks: {len(chunked_documents)}")
print(f"Web documents: {len(webdocs)}")

Source documents: 275
Indexed chunks: 523
Web documents: 214


In [15]:
test_question = "what is black shank on tobacco and how to manage it"

retrieved_docs = retriever.invoke(test_question)

print(f"Retrieved {len(retrieved_docs)} chunks\n")

for index, document in enumerate(retrieved_docs, start=1):
    print("=" * 100)
    print(f"Result {index}")
    print(f"Source: {document.metadata.get('source', 'unknown')}")
    print(f"Source type: {document.metadata.get('source_type', 'unknown')}")
    print(document.page_content[:2000])

Retrieved 6 chunks

Result 1
Source: https://agritech.tnau.ac.in/crop_protection/tabacco_diseases/tobacco_d3.html
Source type: web
|     |     |     |     |
| --- | --- | --- | --- |
| ![TNAU Agritech Portal](https://agritech.tnau.ac.in/crop_protection/images/index/cp_index.jpg) |
| [Home](https://agritech.tnau.ac.in/index.html "Home") \| [About Us](https://agritech.tnau.ac.in/about_us/abt_us.html "About Us") \| [Success Stories](https://agritech.tnau.ac.in/success_stories/success_index.html "Success Stories") \| [Farmers Association](https://agritech.tnau.ac.in/farm_association/farm_assoct.html "Farmer's Association") \| [Farmers' Innovation](https://agritech.tnau.ac.in/farm_innovations/farm_innovations.html) \| [Publications](https://agritech.tnau.ac.in/tnau_publish.html "TNAU Publications") \| [Contact](https://agritech.tnau.ac.in/contact.html "Contact Us") |  |
| Agricultural crops :: Cash crops :: Tobacco

**Black shank: _Phytophthora parasitica_ var _. nicotianae_**

|     |     

In [16]:
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are an agriculture assistant. Answer the user's question using only the supplied context.
If the context does not contain the answer, say that you do not have enough information.
Keep the answer clear and practical. Do not invent facts.

Context:
{context}""",
    ),
    ("human", "{question}"),
])


def ask_rag(question, chat_history=None):
    """Retrieve relevant agriculture documents and answer a user question."""
    if not question or not question.strip():
        raise ValueError("Please enter a non-empty question.")

    documents = retriever.invoke(question)
    context = "\n\n---\n\n".join(document.page_content for document in documents)

    messages = []
    if chat_history:
        messages.extend(chat_history)
    messages.extend(rag_prompt.format_messages(context=context, question=question))

    response = llm.invoke(messages)
    answer = response.content if hasattr(response, "content") else str(response)
    sources = [
        document.metadata.get("source")
        or document.metadata.get("url")
        or document.metadata.get("file_path")
        or document.metadata.get("source_type", "unknown")
        for document in documents
    ]

    return {
        "answer": answer,
        "sources": list(dict.fromkeys(sources)),
        "documents": documents,
    }


chat_history = []
result = ask_rag("what is black shank on tobacco and how to manage it")
print(result["answer"])
print("\nSources:")
for source in result["sources"]:
    print(f"- {source}")

**What is Black Shank on Tobacco?**

Black shank is a serious disease of tobacco caused by the soil-inhabiting fungus *Phytophthora parasitica* var. *nicotianae*. It occurs sporadically in all types of tobacco but causes more damage in areas with high rainfall or irrigated conditions, particularly in light soils in Karnataka, Andhra Pradesh, and Gujarat.

**Symptoms:**
*   **Nursery:** Affected seedlings show blackening of roots and stems near the soil line. Under wet conditions, conidial infection can spread to leaves, causing water-soaked lesions, leaf blight, and collapse.
*   **Field:** Blackening starts at the collar region and spreads both upwards and downwards, causing rotting and necrosis of the entire stem and basal leaves.
*   **First Signs:** The first visible signs in vigorous plants are yellowing of leaves, sudden wilting, and eventual death under warm weather.
*   **Distinctive Feature:** When stems of affected plants are split open, they reveal brown to black dried pith 